<a href="https://colab.research.google.com/github/akshin-kuvshin/numerical-methods/blob/main/lab10/lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №10. Вычисление интеграла методами Монте-Карло

## Выполнил: Данила Аксёнов, гр. 932420

## Вариант: 1

## **Задание 15**

* Вычислить $ \int_{a}^{b} f(x) \, dx $ с точностью $ \epsilon = 0.01 $ методами Монте-Карло: простейшим, геометрическим;
* Вывести значение интеграла и значение $ N $, которое потребовалось для вычисления интеграла с заданной точностью.

In [1]:
import numpy as np
from matplotlib import pyplot as plt
print('Setup completed')

Setup completed


### Исходные данные:

In [2]:
a = 2
b = 3
f = lambda x: np.sqrt(x ** 2 + 2 * x + np.log10(x))
eps = 0.01

#### Дополнительные данные для простейшего метода Монте-Карло:

In [3]:
ft = lambda t: f(a + (b - a) * t)
ft2 = lambda t: ft(t) ** 2
rng = np.random.default_rng(seed=42)
H1 = 1_000

#### Дополнительные данные для геометрического метода Монте-Карло:

In [4]:
# очевидно, что для моего варианта функция f(x) = sqrt(x^2 + 2x + lgx)
# монотонно возрастает на [2; 3] (более того - она монотонно возрастает
# на всей своей области определения) =>
c = f(a) # минимум f на [a; b]
d = f(b) # максимум f на [a; b]
c, d

(np.float64(2.8811508109892445), np.float64(3.934097260455016))

In [5]:
g = lambda x: (f(x) - c) / (d - c)
gt = lambda t: g(a + (b - a) * t)

good = lambda pair: gt(pair[0]) < pair[1]

H2 = 10_000
alpha = 0.01

### Решение:

### **1. Простейший метод Монте-Карло**

В простейшем методе Монте-Карло значение интеграла $ I = \int_a^b f(x) \, dx $ будем аппроксимировать так: $ I \approx \frac{1}{N} \sum_{k=1}^N f(M_k) $, $ - $ где точки $ M_k $ имеют вид $ M_k = a + (b - a) * \tau_k $, где $ \tau_1, \tau_2, \dots, \tau_N $ $ - $ реализации _равномерно распределённой на $ [0; 1] $ случайной величины_. При этом, в целях _уменьшения погрешности округления_, вместо использования рекуррентного соотношения из учебника (см. стр. 176) я буду отдельно хранить сумму $ f(M_1) + f(M_2) + \dots + f(M_N) $ и дополнять её значениями функции $ f $ в новых $ H $ точках на каждой новой итерации, таким образом избежав необходимости хранить все $ N $ реализаций СВ $ \tau $ одновременно и повторно вычислять значения функции $ f $ в каждой из них. Здесь под $ H $ я подразумеваю постоянный шаг увеличения общего количества точек $ N $. Для простейшего метода Монте-Карло я положил $ H = 1\,000 $ и $ N_0 = H $.

Условием остановки, согласно _правилу трёх сигм_, является $ 3 \sqrt{\frac{D(M)}{N}} \le \epsilon $, где $ \epsilon $ $ - $ заданная точность вычисления интеграла, а $ D(M) $ $ - $ _выборочная дисперсия_, вычисляемая по формуле $ D(M) = \frac{1}{N} \sum_{k=1}^N f^2(M_k) - I^2 $, причём вместо точного значения интеграла мы будем брать его текущую аппроксимацию $ I \approx \frac{1}{N} \sum_{k=1}^N f(M_k) $. Опять же, сумму $ f^2(M_1) + f^2(M_2) + \dots + f^2(M_N) $ я буду хранить отдельно и лишь дополнять новыми значениями, что в определённой степени повысит точность вычислений и значительно повысит удобство и читаемость программной реализации метода.

In [6]:
N = H1
M = rng.random(N) # N первых точек
SI = ft(M).sum()
I = SI / N
SD = ft2(M).sum()
D = SD / N - I ** 2

while 3 * np.sqrt(D / N) > eps:
    N += H1
    M = rng.random(H1) # H1 новых точек
    SI += ft(M).sum()
    I = SI / N
    SD += ft2(M).sum()
    D = SD / N - I ** 2

In [7]:
I, N

(np.float64(3.4080173461182337), 9000)

Простейшему методу Монте-Карло хватило $ \mathbf{N = 9\,000} $ точек, чтобы достигнуть заданной точности $ \epsilon = 0.01 $. Полученное значение интеграла $ \mathbf{I = \int_a^b f(x)\,dx = \int_2^3 \sqrt{x^2 + 2x + \lg(x)} \, dx \approx 3.4} $ согласуется с истиной.

---

### **2. Геометрический метод Монте-Карло**

В геометрическом методе Монте-Карло идея другая: пусть $ c $ $ - $ минимальное значение функции $ f $ на отрезке $ [a; b] $, а $ d $ $ - $ максимальное. Для моего варианта, как я посчитал выше (см. п. _"Дополнительные данные для геометрического метода Монте-Карло"_), $ c \approx 2.88115 $, $ d \approx 3.93410 $. Тогда заменами $ x = a + (b - a) \tau $ и $ g = \frac{f - c}{d - c} $ исходную функцию из произвольного прямоугольника размера $ (b - a) \times (d - c) $ в координатах $ (x, f) $ можно отобразить в единичный квадрат, расположенный в I четверти и прилежащий к координатным осям $ O \tau $ и $ O g $ (т.е. с вершинами $ (0, 0), (0, 1), (1, 0), (1, 1) $). Пусть $ I $ $ - $ значение исходного интеграла, которое мы и пытаемся вычислить, а $ I^* $ $ - $ значение интеграла в отображённом единичном квадрате в координатах $ (\tau, g) $. Тогда, согласно виду замен, имеем $ I = (d - c) (b - a) I^* + c (b - a) $.

Интеграл $ I^* $ будем считать следующим образом: возьмём $ N $ пар точек $ (\xi, \eta) $, где $ \xi $ и $ \eta $ $ - $ _независимые равномерно распределённые на $ [0; 1] $ СВ_. Будем считать пару точек _хорошей_, если выполнено $ g(\xi) < \eta $. Пусть $ \nu $ $ - $ количество хороших точек. Тогда можно утверждать, что $ I^* \approx \frac{\nu}{N} $, причём, конечно, согласно _закону больших чисел_, $ \lim_{N \to \infty} \frac{\nu(N)}{N} = I^* $, поэтому, как и в простейшем методе Монте-Карло, будем итеративно увеличивать количество "бросаемых в квадрат" точек. Как было отмечено в учебнике, геометрический метод Монте-Карло требует значительно большее число испытаний, чем простейший, поэтому здесь я положил $ H = 10\,000 $ и, как и в прошлый раз, $ N_0 = H $.

Условие остановки

$$ \frac{1}{2 \sqrt{\alpha N}} \le \epsilon \quad (1)$$

суть следствие _неравенства Чебышева_. Здесь $ \alpha $ $ - $ вероятность ошибки (невыполнения неравенства (1)), поэтому берётся малым. Я, как было предложено в учебнике, положил $ \alpha = 0.01 $.

<u>_Интересный факт_</u>: от итерации к итерации нам не нужно хранить или хотя бы считать текущее аппроксимированное значение интеграла $ I^* $ $ - $ достаточно **накапливать** количество хороших точек $ \nu $.

In [8]:
N = H2
M = rng.random((2, N)) # N первых пар точек (xi, eta)
v = good(M).sum()

while 1 / (2 * np.sqrt(alpha * N)) > eps:
    N += H2
    M = rng.random((2, H2)) # H2 новых пар точек (xi, eta)
    v += good(M).sum()

In [9]:
I = (d - c) * (b - a) * v / N + c * (b - a)
I, N

(np.float64(3.4033996145668737), 250000)

Геометрическому методу Монте-Карло для достижения заданной точности потребовалось значительно больше точек: $ \mathbf{N = 250\,000} $. Результат вычислений $ \mathbf{I = \int_a^b f(x)\,dx = \int_2^3 \sqrt{x^2 + 2x + \lg(x)} \, dx \approx 3.4} $ совпадает и с результатом, полученным с помощью простейшего метода Монте-Карло, и со всеми результатами предыдущих лабораторных работ (предыдущих заданий; ведь именно этот интеграл я считаю из раза в раз новыми и новыми методами), и, конечно, с _истиной_.